In [ ]:
# ── Cell 1: Mount Drive ──────────────────────────────────────────────
# Run once per session. Grants Colab access to your Google Drive so
# checkpoints and the replay buffer sync automatically.

from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/chess-engine')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('sys.path entry:', PROJECT_ROOT / 'src')

In [ ]:
# ── Cell 2: Install deps ─────────────────────────────────────────────
# Installs from requirements.txt into the Colab runtime.
# Fast on subsequent runs once pip's wheel cache is warm.

!pip install -q -r /content/drive/MyDrive/chess-engine/requirements.txt
print('Dependencies ready.')

In [8]:
# ── Cell 3: Import ───────────────────────────────────────────────────
# Verify GPU availability and import the training entry point.

import torch
from main import train

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found — training will be slow. Use Runtime > Change runtime type.')

Device: cuda
GPU  : NVIDIA A100-SXM4-40GB
VRAM : 42.4 GB


In [ ]:
# ── Cell 4: Train (fresh start) ──────────────────────────────────────
# If resuming after a disconnect, run Cell 5 instead.

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/chess-engine')

train(
    # ── paths ────────────────────────────────────────────────────────
    checkpoints_dir = PROJECT_ROOT / 'checkpoints',
    data_dir        = PROJECT_ROOT / 'data',
    book_path       = PROJECT_ROOT / 'books/gm2001.bin',
    elo_log_path    = PROJECT_ROOT / 'runs/elo_log.csv',
    train_log_path  = PROJECT_ROOT / 'runs/train_log.csv',
    # ── architecture ─────────────────────────────────────────────────
    num_blocks  = 10,
    num_filters = 128,
    # ── self-play ────────────────────────────────────────────────────
    games_per_iter = 100,
    mcts_sims      = 100,
    max_game_moves = 250,
    # ── training ─────────────────────────────────────────────────────
    train_steps  = 200,
    batch_size   = 512,
    min_buffer   = 1_000,
    # ── evaluation ───────────────────────────────────────────────────
    eval_every    = 5,
    eval_games    = 20,
    eval_sims     = 100,
    win_threshold = 0.55,
    # ── loop control ─────────────────────────────────────────────────
    num_iters  = 1_000,
    save_every = 10,
)

In [ ]:
# ── Cell 5: Resume ───────────────────────────────────────────────────
# Run after a Colab disconnect. Keep all hyperparameters identical.

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/chess-engine')

train(
    # ── paths ────────────────────────────────────────────────────────
    checkpoints_dir = PROJECT_ROOT / 'checkpoints',
    data_dir        = PROJECT_ROOT / 'data',
    book_path       = PROJECT_ROOT / 'books/gm2001.bin',
    elo_log_path    = PROJECT_ROOT / 'runs/elo_log.csv',
    train_log_path  = PROJECT_ROOT / 'runs/train_log.csv',
    # ── architecture (must match original run) ────────────────────────
    num_blocks  = 10,
    num_filters = 128,
    # ── self-play ────────────────────────────────────────────────────
    games_per_iter = 100,
    mcts_sims      = 100,
    max_game_moves = 100,
    # ── training ─────────────────────────────────────────────────────
    train_steps  = 200,
    batch_size   = 512,
    min_buffer   = 1_000,
    # ── evaluation ───────────────────────────────────────────────────
    eval_every    = 5,
    eval_games    = 20,
    eval_sims     = 100,
    win_threshold = 0.55,
    # ── loop control ─────────────────────────────────────────────────
    num_iters  = 1_000,
    save_every = 10,
)